# Laboratorio 3 — Task 2

Calibración y validación del modelo SIR con Runge–Kutta de 4.º orden (solo NumPy).

## Parámetros e información inicial

Población total: $N = 500\,000$

Condiciones iniciales (Task 1.1):

$$
I(0)=800,\quad R(0)=200,\quad S(0)=499\,000
$$

Parámetros iniciales:

$$
\gamma = \frac{1}{7},\quad \beta = 0.4 \implies R_0 = \frac{\beta}{\gamma} = 2.8
$$

Sistema SIR:

$$
\frac{dS}{dt}=-\beta\frac{SI}{N},\quad
\frac{dI}{dt}=\beta\frac{SI}{N}-\gamma I,\quad
\frac{dR}{dt}=\gamma I
$$

In [1]:
import numpy as np

N = 500000.0
S0 = 499000.0
I0 = 800.0
R0_inicial = 200.0

gamma = 1.0 / 7.0
beta_inicial = 0.4

estado_inicial = np.array([S0, I0, R0_inicial], dtype=float)

dias_observados = np.array([7.0, 14.0, 21.0, 28.0])
I_observados = np.array([1850.0, 4200.0, 8900.0, 16400.0])

print(f"N = {N:.0f}")
print(f"S0, I0, R0 = {S0:.0f}, {I0:.0f}, {R0_inicial:.0f}")
print(f"gamma = {gamma:.6f}, beta_inicial = {beta_inicial}, R0 = {beta_inicial / gamma:.2f}")

N = 500000
S0, I0, R0 = 499000, 800, 200
gamma = 0.142857, beta_inicial = 0.4, R0 = 2.80


## Modelo SIR + RK4

In [2]:
def sir(estado, beta, gamma, N):
    S, I, R = estado
    nuevos_infectados = beta * S * I / N
    nuevos_recuperados = gamma * I
    return np.array([
        -nuevos_infectados,
        nuevos_infectados - nuevos_recuperados,
        nuevos_recuperados,
    ], dtype=float)


def rk4(beta, gamma, t_final, dt=0.1):
    pasos = int(round(t_final / dt))
    tiempos = np.linspace(0.0, t_final, pasos + 1)
    estados = np.zeros((pasos + 1, 3), dtype=float)
    estados[0] = estado_inicial

    for i in range(pasos):
        y = estados[i]
        h = tiempos[i + 1] - tiempos[i]

        k1 = sir(y, beta, gamma, N)
        k2 = sir(y + (h / 2.0) * k1, beta, gamma, N)
        k3 = sir(y + (h / 2.0) * k2, beta, gamma, N)
        k4 = sir(y + h * k3, beta, gamma, N)

        estados[i + 1] = y + (h / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

    return tiempos, estados


def infectados_semanales(beta):
    tiempos, estados = rk4(beta=beta, gamma=gamma, t_final=28.0, dt=0.1)
    return np.interp(dias_observados, tiempos, estados[:, 1])


def calcular_sce(beta):
    I_modelo = infectados_semanales(beta)
    return np.sum((I_observados - I_modelo) ** 2)

---
# Task 2.1

## a) Simulación inicial y SCE

Se ejecuta el modelo con $\beta=0.4$, $\gamma=1/7$ y $\Delta t=0.1$ días.

$$
SCE=\sum_{k=1}^{4}(I_{\text{obs},k}-I_{\text{modelo},k})^2
$$

In [11]:
I_inicial_modelo = infectados_semanales(beta_inicial)
errores_iniciales = I_observados - I_inicial_modelo
sce_inicial = np.sum(errores_iniciales ** 2)

print("MODELO CON PARÁMETROS DEL TASK 1")
print()
print(f"{'Semana':>6} {'I_obs':>10} {'I_modelo':>12} {'Error':>12}")
print("-" * 44)
for semana in range(4):
    print(
        f"{semana + 1:>6} "
        f"{I_observados[semana]:>10.2f} "
        f"{I_inicial_modelo[semana]:>12.2f} "
        f"{errores_iniciales[semana]:>12.2f}"
    )

print()
print(f"SCE inicial = {sce_inicial:,.2f}")
print()



MODELO CON PARÁMETROS DEL TASK 1

Semana      I_obs     I_modelo        Error
--------------------------------------------
     1    1850.00      4753.15     -2903.15
     2    4200.00     25757.21    -21557.21
     3    8900.00     92842.82    -83942.82
     4   16400.00    137455.37   -121055.37

SCE inicial = 22,173,941,506.89



Observación: con beta=0.4 el modelo SOBREESTIMA I(t) en las cuatro semanas, por lo que la calibración debe reducir beta.


## b) Hipótesis y calibración de $\beta$

El término $\beta\,SI/N$ controla los nuevos infectados. Como $\beta=0.4$ produce demasiados infectados, la hipótesis es:

$$
\beta^* < 0.4
$$

Se mantiene $\gamma=1/7$ y se busca el $\beta$ que minimiza $SCE(\beta)$.

In [4]:
# Búsqueda amplia
betas = np.linspace(0.05, 0.80, 7501)
sces = np.array([calcular_sce(b) for b in betas])
beta_aproximado = betas[np.argmin(sces)]

# Refinamiento local
betas_fin = np.linspace(beta_aproximado - 0.0025, beta_aproximado + 0.0025, 5001)
sces_fin = np.array([calcular_sce(b) for b in betas_fin])
indice_final = np.argmin(sces_fin)

beta_estrella = betas_fin[indice_final]
sce_calibrado = sces_fin[indice_final]
I_calibrado = infectados_semanales(beta_estrella)
errores_calibrados = I_observados - I_calibrado
R0_estrella = beta_estrella / gamma

print("MODELO CALIBRADO")
print()
print(f"beta* = {beta_estrella:.6f} día^-1")
print(f"R0*  = {R0_estrella:.6f}")
print(f"SCE* = {sce_calibrado:,.2f}")
print()
print(f"{'Semana':>6} {'I_obs':>10} {'I_calibrado':>12} {'Error':>12}")
print("-" * 44)
for semana in range(4):
    print(
        f"{semana + 1:>6} "
        f"{I_observados[semana]:>10.2f} "
        f"{I_calibrado[semana]:>12.2f} "
        f"{errores_calibrados[semana]:>12.2f}"
    )

MODELO CALIBRADO

beta* = 0.257428 día^-1
R0*  = 1.801996
SCE* = 606,066.69

Semana      I_obs  I_calibrado        Error
--------------------------------------------
     1    1850.00      1771.47        78.53
     2    4200.00      3875.81       324.19
     3    8900.00      8263.14       636.86
     4   16400.00     16698.69      -298.69


In [5]:
rmse = np.sqrt(np.mean(errores_calibrados ** 2))
mape = np.mean(np.abs(errores_calibrados / I_observados)) * 100.0
ss_total = np.sum((I_observados - np.mean(I_observados)) ** 2)
r2 = 1.0 - sce_calibrado / ss_total

print("MÉTRICAS DE AJUSTE")
print()
print(f"RMSE = {rmse:.2f}")
print(f"MAPE = {mape:.2f}%")
print(f"R^2  = {r2:.6f}")
print()
print(f"Reducción SCE: {sce_inicial:,.2f} → {sce_calibrado:,.2f}")

MÉTRICAS DE AJUSTE

RMSE = 389.25
MAPE = 5.24%
R^2  = 0.995094

Reducción SCE: 22,173,941,506.89 → 606,066.69


## c) Análisis de sensibilidad

Se simulan tres escenarios con $\gamma$ fijo:

$$
\beta \in \{0.8\beta^*,\ \beta^*,\ 1.2\beta^*\}
$$

Hasta $t_f=365$ días. El tamaño final de la epidemia es $N - S(t_f)$.

In [10]:
betas_sensibilidad = np.array([
    0.8 * beta_estrella,
    beta_estrella,
    1.2 * beta_estrella,
])
etiquetas = ["0.8 β*", "β*", "1.2 β*"]

resultados_pico = np.zeros(3)
resultados_tiempo = np.zeros(3)
resultados_final = np.zeros(3)

print("ANÁLISIS DE SENSIBILIDAD")
print()
print(f"{'Escenario':>10} {'beta':>12} {'Pico I(t)':>14} {'t_pico':>10} {'Tamaño final':>14}")
print("-" * 64)

for i, beta in enumerate(betas_sensibilidad):
    tiempos, estados = rk4(beta=beta, gamma=gamma, t_final=365.0, dt=0.1)
    S = estados[:, 0]
    I = estados[:, 1]

    indice_pico = np.argmax(I)
    resultados_pico[i] = I[indice_pico]
    resultados_tiempo[i] = tiempos[indice_pico]
    resultados_final[i] = N - S[-1]

    print(
        f"{etiquetas[i]:>10} "
        f"{beta:>12.6f} "
        f"{resultados_pico[i]:>14.2f} "
        f"{resultados_tiempo[i]:>9.1f}d "
        f"{resultados_final[i]:>14.2f}"
    )

pico_base = resultados_pico[1]
variacion_menos_20 = ((resultados_pico[0] - pico_base) / pico_base) * 100.0
variacion_mas_20 = ((resultados_pico[2] - pico_base) / pico_base) * 100.0

print()
print("VARIACIÓN DEL PICO")
print(f"Con beta -20%: {variacion_menos_20:.2f}%")
print(f"Con beta +20%: {variacion_mas_20:.2f}%")
print()


ANÁLISIS DE SENSIBILIDAD

 Escenario         beta      Pico I(t)     t_pico   Tamaño final
----------------------------------------------------------------
    0.8 β*     0.205942       26800.40      79.4d      272259.98
        β*     0.257428       59484.55      52.0d      366920.08
    1.2 β*     0.308914       90713.16      39.1d      418128.25

VARIACIÓN DEL PICO
Con beta -20%: -54.95%
Con beta +20%: 52.50%



El modelo es sensible a beta. Un cambio de ±20% en la tasa de transmisión provoca cambios >50% en el pico de infectados.


---
# Task 2.2

## a) Validación del modelo calibrado

Se aplican tres criterios verificables:

### 1. Validez estructural

El modelo divide la población en $S \rightarrow I \rightarrow R$ con flujos $\beta SI/N$ e $\gamma I$.  
Con los datos disponibles (S, I, R y período infeccioso) esta estructura es coherente; no hay evidencia de un período de latencia que justifique un compartimento expuesto.



### 2. Validez matemática interna

Debe conservarse $N = S + I + R$ y los compartimentos deben permanecer no negativos. Unidades: $[\beta]=[\gamma]=\text{día}^{-1}$.

In [7]:
tiempos, estados = rk4(beta=beta_estrella, gamma=gamma, t_final=365.0, dt=0.1)

poblacion_total = np.sum(estados, axis=1)
error_conservacion = np.max(np.abs(poblacion_total - N))
minimos = estados.min(axis=0)

print("CONSERVACIÓN DE POBLACIÓN")
print(f"Error máximo |S + I + R - N| = {error_conservacion:.10e}")
print()
print("NO NEGATIVIDAD")
print(f"mín(S, I, R) = {minimos[0]:.6f}, {minimos[1]:.6f}, {minimos[2]:.6f}")
print()
print("Resultado: el modelo CUMPLE la validez matemática interna.")

CONSERVACIÓN DE POBLACIÓN
Error máximo |S + I + R - N| = 1.1641532183e-09

NO NEGATIVIDAD
mín(S, I, R) = 133079.922873, 0.000014, 200.000000

Resultado: el modelo CUMPLE la validez matemática interna.


### 3. Validez empírica

Con $\beta=0.4$: $SCE \approx 2.22\times 10^{10}$.  
Con $\beta^*$: $SCE^* \approx 6.06\times 10^{5}$, RMSE $\approx 389$, MAPE $\approx 5.24\%$, $R^2 \approx 0.995$.

**Resultado:** el modelo calibrado **cumple razonablemente** este criterio para los cuatro datos usados.  
Limitación: no hay validación con datos independientes (los mismos puntos se usaron para calibrar).

In [8]:
print("VALIDEZ EMPÍRICA (resumen numérico)")
print(f"SCE (beta=0.4) = {sce_inicial:,.2f}")
print(f"SCE (beta*)    = {sce_calibrado:,.2f}")
print(f"RMSE = {rmse:.2f}")
print(f"MAPE = {mape:.2f}%")
print(f"R^2  = {r2:.6f}")

VALIDEZ EMPÍRICA (resumen numérico)
SCE (beta=0.4) = 22,173,941,506.89
SCE (beta*)    = 606,066.69
RMSE = 389.25
MAPE = 5.24%
R^2  = 0.995094


## b) Limitación estructural y posible extensión

Limitación importante: el modelo supone $\beta$ **constante** durante toda la epidemia.

En la realidad, el comportamiento (distanciamiento, mascarillas, políticas, etc.) puede cambiar la intensidad efectiva de transmisión. Que $\beta=0.4$ (Task 1) no ajuste las primeras semanas, mientras $\beta^*\approx 0.257$ sí lo hace, sugiere que la transmisión efectiva observada es menor.

### Extensión propuesta: $\beta(t)$

$$
\frac{dS}{dt}=-\beta(t)\frac{SI}{N},\quad
\frac{dI}{dt}=\beta(t)\frac{SI}{N}-\gamma I,\quad
\frac{dR}{dt}=\gamma I
$$

Esto corrige la suposición de contacto constante sin añadir compartimentos para los que no hay datos. Un SEIR o una población abierta serían útiles solo si hubiera evidencia de latencia o demografía relevante.

---
# Conclusiones

1. Con los parámetros del Task 1 ($\beta=0.4$), el modelo **sobreestima** los infectados de las primeras 4 semanas ($SCE \sim 10^{10}$).
2. Calibrando solo $\beta$ (con $\gamma=1/7$ fijo) se obtiene $\beta^* \approx 0.257428$ día$^{-1}$ y $R_0^* \approx 1.802$, con $SCE^* \sim 6\times 10^5$.
3. El modelo es **sensible** a $\beta$: $\pm 20\%$ en transmisión cambia el pico de $I(t)$ en torno a $-55\%$ / $+52\%$.
4. El ajuste a los cuatro datos es bueno, pero la hipótesis de $\beta$ constante puede ser restrictiva; una extensión natural es $\beta(t)$.

In [9]:
print("RESUMEN FINAL")
print(f"beta*  = {beta_estrella:.6f}")
print(f"R0*    = {R0_estrella:.6f}")
print(f"SCE*   = {sce_calibrado:,.2f}")
print(f"Δ pico (-20% beta) = {variacion_menos_20:.2f}%")
print(f"Δ pico (+20% beta) = {variacion_mas_20:.2f}%")

RESUMEN FINAL
beta*  = 0.257428
R0*    = 1.801996
SCE*   = 606,066.69
Δ pico (-20% beta) = -54.95%
Δ pico (+20% beta) = 52.50%
